In [2]:
import glob
import re
import pandas as pd
import shutil
from tqdm import tqdm

In [6]:
files = glob.glob("../done/K2211*")
f_name = []
for f in files:
    p = 'done/(K\d{6}).TXT'
    f_name.append(re.search(p,f).group(1))

In [7]:
len(f_name)

26

In [20]:
def mkcsv(FILE_NAME):
    with open('../done/' + FILE_NAME + '.TXT', encoding='shift-jis') as f:
        data = f.readlines()
    racers = []
    race = []
    place = []
    odds = []
    for i in data:
        if  re.match('^\s{3}第.+日', i):
            place.append(i.replace('\u3000','').replace('\n','').replace('/','_').replace('_ ', '_'))
        elif re.match('^\s+\d+R.+\s+H', i):
            race.append(i.replace('\u3000','').replace('\n',''))
        elif re.match('^\s{2}0[1-9]\s', i):
            racers.append(i.replace('\u3000','').replace('\n',''))
        elif re.match('^\s{5}.*[1-9\s][0-9]R', i):
            odds.append(i)

    pattern_place = re.compile('^.+(\d{4}_.+\d+)\s.+ボートレース([^0-9A-Z\s-]+)')
    pl = [re.match(pattern_place, i).groups() for i in place]

    pattern_race = re.compile('^\s+(\d+R)\s+.+\s{2}([^0-9A-Z\s-]+)\s{2}風\s{2}([^0-9A-Z\s]+).+(\d+)m.+波.+(\d+)cm')
    rc = [re.match(pattern_race, i).groups() for i in race]

    pattern_racers = re.compile('^\s{2}0([1-6])\s{2}([1-6])\s(\d{4})\s([^0-9]+)\s(\d+)\s+(\d+)\s+(\d+.\d{2})\s{3}(\d)\s{3}[\sF](\d.\d{2})\s+')
    rs = [re.match(pattern_racers, i).groups() for i in racers]

    pattern_odds = re.compile('^\s{5}.+(\d+R)\s{2}(\d-\d-\d|[^\s]*)\s*(\d+|\s)\s*(\d-\d-\d|[^\s]*)\s*(\d+|\s)\s+(\d-\d|[^\s]*)\s+(\d+|\s)\s+(\d-\d|[^\s]*)\s+(\d+|\s)')
    od = [re.match(pattern_odds, i).groups() for i in odds]

    column_rs = ['着','艇番','選手登番','選手名','モーター','ボート','展示','進入','スタートタイミング']
    df_rs = pd.DataFrame(rs, columns=column_rs)

    column_rc = ['R','天気','風向','風量','波']
    df_rc = pd.DataFrame(rc, columns=column_rc)

    column_pl = ['日時','場所']
    df_pl = pd.DataFrame(pl, columns=column_pl)
    df_rc['場所'] = ''
    df_rc['日時'] = ''

    column_od = ['R', '3連単_結果', '3連単_払戻', '3連複_結果', '3連複_払戻', '2連単_結果', '2連単_払戻', '2連複_結果', '2連複_払戻']
    df_od = pd.DataFrame(od, columns=column_od)
    for i in column_od:
        if i != "R":
            df_rc[i] = ''
    m = -1
    o = 0
    for i in range(df_rc.shape[0]):
        l = '1R'
        if i > 0:
            l = df_rc['R'][i-1]
        n = df_rc['R'][i]
        if n == '1R':
            m = m + 1
            df_rc['場所'][i] = df_pl['場所'][m]
            df_rc['日時'][i] = df_pl['日時'][m]
            if l != n:
                o = o +1
            for j in column_od:
                if j == 'R':
                    pass
                else:
                    df_rc[j][i] = df_od[j][o]

        else:
            df_rc['場所'][i] = df_pl['場所'][m]
            df_rc['日時'][i] = df_pl['日時'][m]
            if l != n:
                o = o +1
            for j in column_od:
                if j == 'R':
                    pass
                else:
                    df_rc[j][i] = df_od[j][o]
    
    m = -1
    df_rs.loc[:,'場所'] = ''
    for c in df_rc.columns:
        df_rs.loc[:,c] = ''
    for i in range(df_rs.shape[0]):
        n = int(df_rs['着'][i])
        if n == 1  :
            m = m + 1
            for c in range(len(df_rc.columns)):
                df_rs.iloc[i,c] = df_rc.iloc[m,c]
        else:
            for c in range(len(df_rc.columns)):
                df_rs.iloc[i,c] = df_rc.iloc[m,c]
    
    place_code = {'桐生':'01','戸田':'02','江戸川':'03','平和島':'04','多摩川':'05','浜名湖':'06','蒲郡':'07','常滑':'08','津':'09','三国':'10','びわこ':'11','琵琶湖':'11','住之江':'12','尼崎':'13','鳴門':'14','丸亀':'15','児島':'16','宮島':'17','徳山':'18','下関':'19','若松':'20','芦屋':'21','福岡':'22','唐津':'23','大村':'24'}

    df_rs['場所'] = df_rs['場所'].map(place_code)
    df_rs['RaceID'] = df_rs['日時'] + '_' + df_rs['場所'] + '_' + df_rs['R']

    df_rs.to_csv('../csv/' + FILE_NAME +  '.csv')
    #shutil.move('../data/' + FILE_NAME + '.TXT', '../done/' + FILE_NAME + '.TXT')

In [9]:
df = pd.read_csv('../csv/K190101.csv')

In [21]:
for f in tqdm(f_name):
    print(f)
    mkcsv(f)


  0%|          | 0/26 [00:00<?, ?it/s]

K221104
      R 3連単_結果 3連単_払戻 3連複_結果 3連複_払戻 2連単_結果 2連単_払戻 2連複_結果 2連複_払戻
0    1R  1-3-2   1010  1-2-3    430    1-3    210    1-3    200
1    2R  1-4-3   1530  1-3-4    450    1-4    450    1-4    160
2    3R  2-4-6   2510  2-4-6    360    2-4   3050    2-4   1050
3    4R  1-2-3   1680  1-2-3    600    1-2    400    1-2    190
4    5R  1-3-5   2650  1-3-5   1060    1-3    780    1-3    760
..   ..    ...    ...    ...    ...    ...    ...    ...    ...
139  8R  3-1-5   2380  1-3-5    900    3-1    600    1-3    390
140  9R  6-3-1   4900  1-3-6    460    6-3   2260    3-6    750
141  0R  1-2-5   1920  1-2-5    680    1-2    360    1-2    200
142  1R  2-3-4   7440  2-3-4   2390    2-3   1900    2-3   1700
143  2R  6-2-1  21470  1-2-6    780    6-2   5450    2-6   2480

[144 rows x 9 columns]


  4%|▍         | 1/26 [00:02<01:00,  2.43s/it]

K221110
      R 3連単_結果 3連単_払戻 3連複_結果 3連複_払戻 2連単_結果 2連単_払戻 2連複_結果 2連複_払戻
0    1R  6-1-3  14860  1-3-6    610    6-1   5450    1-6    880
1    2R  4-1-5   7310  1-4-5    910    4-1   1180    1-4    630
2    3R  1-2-5   4640  1-2-5   1620    1-2   1000    1-2    670
3    4R  4-1-5   4420  1-4-5    930    4-1   1190    1-4    440
4    5R  1-3-6    650  1-3-6    280    1-3    300    1-3    210
..   ..    ...    ...    ...    ...    ...    ...    ...    ...
139  8R  1-5-3    780  1-3-5    280    1-5    290    1-5    160
140  9R  2-1-6   1390  1-2-6    490    2-1    410    1-2    140
141  0R  5-1-3  21670  1-3-5   1760    5-1   3770    1-5    740
142  1R  1-3-5    870  1-3-5    580    1-3    230    1-3    220
143  2R  1-6-2   5020  1-2-6    820    1-6   2400    1-6   1350

[144 rows x 9 columns]


  8%|▊         | 2/26 [00:05<01:01,  2.55s/it]

...

 15%|█▌        | 4/26 [00:09<00:48,  2.22s/it]

K221113
      R 3連単_結果 3連単_払戻 3連複_結果 3連複_払戻 2連単_結果 2連単_払戻 2連複_結果 2連複_払戻
0    1R  1-2-5    740  1-2-5    440    1-2    200    1-2    160
1    2R  1-6-2   4340  1-2-6    610    1-6   2180    1-6   2060
2    3R  2-4-6  22890  2-4-6   5220    2-4   4940    2-4   3100
3    4R  4-6-1   3620  1-4-6    820    4-6   1220    4-6   1420
4    5R  5-1-6  28980  1-5-6   5640    5-1   4290    1-5   1760
..   ..    ...    ...    ...    ...    ...    ...    ...    ...
139  8R  1-5-4   1180  1-4-5    570    1-5    320    1-5    360
140  9R  4-1-5   3790  1-4-5   1590    4-1   1100    1-4    320
141  0R  1-5-3   1430  1-3-5    390    1-5    650    1-5    280
142  1R  1-5-3   1130  1-3-5    370    1-5    330    1-5    290
143  2R  1-2-3   1300  1-2-3    540    1-2    370    1-2    320

[144 rows x 9 columns]


 19%|█▉        | 5/26 [00:11<00:48,  2.33s/it]


K221107


AttributeError: 'NoneType' object has no attribute 'groups'